In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Big Dataset EDA").getOrCreate()

data_url = "https://raw.githubusercontent.com/jpatokal/openflights/master/data/airports.dat"
!wget -q $data_url -O airports.csv

df = spark.read.csv("airports.csv", header=False, inferSchema=True)

df = df.toDF("airport_id", "name", "city", "country", "IATA", "ICAO", "lat", "lon", "alt", "timezone", "dst", "tz_db", "type", "source")

df.show(5)

df.printSchema()

+----------+--------------------+------------+----------------+----+----+------------------+------------------+----+--------+---+--------------------+-------+-----------+
|airport_id|                name|        city|         country|IATA|ICAO|               lat|               lon| alt|timezone|dst|               tz_db|   type|     source|
+----------+--------------------+------------+----------------+----+----+------------------+------------------+----+--------+---+--------------------+-------+-----------+
|         1|      Goroka Airport|      Goroka|Papua New Guinea| GKA|AYGA|-6.081689834590001|     145.391998291|5282|      10|  U|Pacific/Port_Moresby|airport|OurAirports|
|         2|      Madang Airport|      Madang|Papua New Guinea| MAG|AYMD|    -5.20707988739|     145.789001465|  20|      10|  U|Pacific/Port_Moresby|airport|OurAirports|
|         3|Mount Hagen Kagam...| Mount Hagen|Papua New Guinea| HGU|AYMH|-5.826789855957031|144.29600524902344|5388|      10|  U|Pacific/Port_Mor

In [4]:
df.describe("lat", "lon").show()

+-------+------------------+-------------------+
|summary|               lat|                lon|
+-------+------------------+-------------------+
|  count|              7698|               7698|
|   mean| 25.80844248491244|-1.3905462050706061|
| stddev|28.404945978721155|  86.51916220166679|
|    min|             -90.0|     -179.876998901|
|    max|              89.5|      179.951004028|
+-------+------------------+-------------------+



In [6]:
from pyspark.sql import functions as F

df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c)
          for c in df.columns]).show()

+----------+----+----+-------+----+----+---+---+---+--------+---+-----+----+------+
|airport_id|name|city|country|IATA|ICAO|lat|lon|alt|timezone|dst|tz_db|type|source|
+----------+----+----+-------+----+----+---+---+---+--------+---+-----+----+------+
|         0|   0|  49|      0|   0|   0|  0|  0|  0|       0|  0|    0|   0|     0|
+----------+----+----+-------+----+----+---+---+---+--------+---+-----+----+------+



In [12]:
# Count rows per country and show highest first
country_counts = df.groupBy("country").count().orderBy(F.desc("count"))
country_counts.show(10)

+--------------+-----+
|       country|count|
+--------------+-----+
| United States| 1512|
|        Canada|  430|
|     Australia|  334|
|        Russia|  264|
|        Brazil|  264|
|       Germany|  249|
|         China|  241|
|        France|  217|
|United Kingdom|  167|
|         India|  148|
+--------------+-----+
only showing top 10 rows


In [13]:
df.createOrReplaceTempView("airports")

In [15]:
spark.sql(
"""
    SELECT country, COUNT(*) AS airport_count
    FROM airports
    GROUP BY country
    ORDER BY airport_count DESC
    LIMIT 10
"""
).show()

+--------------+-------------+
|       country|airport_count|
+--------------+-------------+
| United States|         1512|
|        Canada|          430|
|     Australia|          334|
|        Russia|          264|
|        Brazil|          264|
|       Germany|          249|
|         China|          241|
|        France|          217|
|United Kingdom|          167|
|         India|          148|
+--------------+-------------+



In [17]:
spark.sql(
"""
    SELECT name, city, IATA, ICAO
    FROM airports
    WHERE country='Sri Lanka'
"""
).show()

+--------------------+----------------+----+----+
|                name|            city|IATA|ICAO|
+--------------------+----------------+----+----+
|Bandaranaike Inte...|         Colombo| CMB|VCBI|
|Anuradhapura Air ...|    Anuradhapura| ACJ|VCCA|
|  Batticaloa Airport|      Batticaloa| BTC|VCCB|
|Colombo Ratmalana...|         Colombo| RML|VCCC|
|      Ampara Airport|          Galoya| ADP|VCCG|
|Kankesanturai Air...|          Jaffna| JAF|VCCJ|
|   China Bay Airport|    Trinciomalee| TRR|VCCT|
|     Koggala Airport|         Koggala| KCT|VCCK|
|   Weerawila Airport|        Wirawila| WRZ|VCCW|
|Mattala Rajapaksa...|         Mattala| HRI|VCRI|
|Sigiriya Air Forc...|        Sigiriya| GIU|VCCS|
|Hingurakgoda Air ...|Polonnaruwa Town| HIM|VCCH|
+--------------------+----------------+----+----+

